# Horus · entrenar la cabeza de objetos — Kaggle, sin token

Pensado para **Save & Run All (Commit)**: lo arrancás y se va solo hasta 12 h.
No pide ninguna API key: D-Fire entra como *dataset adjunto* desde la interfaz.

## Antes de apretar Run — tres cosas en el panel de la derecha

1. **Accelerator → GPU** (T4 o P100, cualquiera sirve).
2. **Internet → On.** Open Images se baja desde acá; sin internet el notebook
   se detiene en la celda 2. Kaggle pide teléfono verificado para habilitarlo.
3. **Add Data → buscá `D-Fire`** y adjuntá el que aparezca. Aporta humo y llama.
   Si no adjuntás ninguno el notebook sigue igual, pero `llama` queda flaca —
   y te avisa. No es grave: el incendio ya lo cubre la cabeza de segmentación
   con F1 99,1 %.

## Qué hace, y qué NO hace

**No reentrena el backbone.** El backbone no se entrena nunca: es un ResNet-50
de ImageNet más un FPN que se sortea al azar y se congela. Lo que se perdió es
un número de la lotería, no un modelo.

Esto sortea un FPN nuevo, entrena una cabeza contra él y **guarda los dos
juntos con huella**, para que no se pueda volver a perder en silencio.

Salís con **un archivo**: `head_best_solo.pt` (~33 MB), autocontenido.
Referencia a batir: el v2, **mAP@0.50 = 0.5335**.


## 1 · Entorno

In [ ]:
import os, sys, subprocess, shutil, pathlib, json
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "SIN GPU")
import torch
if not torch.cuda.is_available():
    raise SystemExit("Sin GPU. Panel derecho -> Accelerator -> GPU, y volvé a correr.")
print("torch", torch.__version__)

TRABAJO = "/kaggle/working"
DATOS   = "/kaggle/temp/ds"      # scratch: no cuenta contra los 20 GB de output
CKPT    = f"{TRABAJO}/checkpoints"
os.makedirs(DATOS, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
print("entradas adjuntas:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "(ninguna)")

## 2 · Internet, y clonar el repo con las correcciones

**Compuerta 1.** Si la rama no trae los tres arreglos del 15/09, esto se
detiene: los anchors del v2, `huella_backbone()` y `--incrustar-backbone`.
Si falla, commiteá y pusheá desde tu máquina.

In [ ]:
import socket, subprocess, os, sys, shutil
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("Sin internet. Panel derecho -> Internet -> On.")

REPO = "https://github.com/Teo50000/Horus-AI.git"
RAMA = "Models"

os.chdir("/kaggle/working")
if os.path.exists("Horus-AI"): shutil.rmtree("Horus-AI")
subprocess.run(["git","clone","--depth","1","-b",RAMA,REPO,"Horus-AI"], check=True)

RAIZ = "/kaggle/working/Horus-AI/horus"
OBJ  = f"{RAIZ}/04_cabezas/objetos"
sys.path[:0] = [f"{RAIZ}/03_backbone", OBJ]

fallas = []
if "(16, 20, 25)" not in open(f"{OBJ}/entrenar_objetos_cuda.py", encoding="utf-8").read():
    fallas.append("entrenar_objetos_cuda.py: anchors del v2")
if "def huella_backbone" not in open(f"{RAIZ}/03_backbone/shared_backbone.py", encoding="utf-8").read():
    fallas.append("shared_backbone.py: huella_backbone()")
if "--incrustar-backbone" not in open(f"{OBJ}/exportar_objetos.py", encoding="utf-8").read():
    fallas.append("exportar_objetos.py: --incrustar-backbone")
if fallas:
    for f in fallas: print("  ✗ falta", f)
    raise SystemExit(f"La rama '{RAMA}' no trae las correcciones del 15/09.")
print("✓ la rama trae las tres correcciones")

## 3 · D-Fire desde el dataset adjunto — acá es donde se evita el token

`bajar_datasets.py` separa *bajar* de *normalizar*: el normalizador lee lo que
haya en `datasets/_crudo/<fuente>/`, sin importar cómo llegó. Así que en vez de
bajar D-Fire con la API de Kaggle, se enlaza el dataset adjunto a esa carpeta y
el resto del pipeline no se entera.

Busca solo, en cualquier entrada adjunta, una carpeta que tenga imágenes y sus
`.txt` al lado o en un `labels/` paralelo — que es el formato YOLO que D-Fire
ya usa (0=smoke, 1=fire, los mismos índices que nosotros).

In [ ]:
import pathlib, os
EXT = {".jpg",".jpeg",".png",".bmp",".webp"}
crudo_dfire = pathlib.Path(OBJ) / "datasets" / "_crudo" / "d-fire"
crudo_dfire.parent.mkdir(parents=True, exist_ok=True)

def parece_yolo(d: pathlib.Path) -> bool:
    imgs = [p for p in d.rglob("*") if p.suffix.lower() in EXT][:40]
    if len(imgs) < 20: return False
    for img in imgs:
        if img.with_suffix(".txt").exists(): return True
        partes = list(img.parts)
        if "images" in partes:
            i = len(partes) - 1 - partes[::-1].index("images")
            partes[i] = "labels"
            if pathlib.Path(*partes).with_suffix(".txt").exists(): return True
    return False

origen = None
for entrada in sorted(pathlib.Path("/kaggle/input").glob("*")) if os.path.exists("/kaggle/input") else []:
    if parece_yolo(entrada):
        origen = entrada; break
    for sub in sorted(p for p in entrada.iterdir() if p.is_dir()):
        if parece_yolo(sub):
            origen = sub; break
    if origen: break

if origen:
    if crudo_dfire.exists() or crudo_dfire.is_symlink(): crudo_dfire.unlink(missing_ok=True)
    crudo_dfire.symlink_to(origen, target_is_directory=True)
    n = sum(1 for p in origen.rglob("*") if p.suffix.lower() in EXT)
    print(f"✓ D-Fire enlazado desde {origen}  ({n} imágenes)")
else:
    print("⚠ no encontré ningún dataset adjunto en formato YOLO.")
    print("  Sigo sin D-Fire: 'llama' va a quedar flaca (segmentación cubre el fuego).")
    print("  Si lo querés: Add Data -> buscá 'D-Fire' -> Add, y volvé a correr.")

## 4 · Dependencias

In [ ]:
%pip install -q fiftyone datasets pycocotools 2>&1 | tail -2
print("listo")

## 5 · Bajar el resto y armar la mezcla

Open Images (persona, pistola, cuchillo, celular, paquete) y Pyro-SDIS (humo):
ninguno pide login. D-Fire se saltea porque ya está enlazado del paso 3.

Es la parte lenta — Open Images baja clase por clase.

In [ ]:
%cd {OBJ}
!python bajar_datasets.py --verificar
!python bajar_datasets.py --descargar pyro-sdis openimages
!python bajar_datasets.py --normalizar
!python bajar_datasets.py --armar

## 6 · Compuerta 2 · contar cajas ANTES de gastar GPU

Hubo un bug en el que **todas** las etiquetas de Open Images traían únicamente
la clase 2 (persona). Entrenar así deja paquete, celular, cuchillo y pistola
sin una sola caja positiva: el modelo aprende que no existen.

La corrección ya está en `bajar_datasets.py`, pero esto lo **cuenta** en vez de
asumirlo. Si falta alguna de las cinco que desbloquean el resto del sistema,
se detiene. `humo` y `llama` solo avisan: sin D-Fire es esperable, y el
incendio no depende de ellas.

In [ ]:
import collections, pathlib
CLASES = ("humo","llama","persona","pistola","cuchillo","celular","paquete")
IMPRESCINDIBLES = {"persona","pistola","cuchillo","celular","paquete"}

raiz = pathlib.Path(OBJ)/"datasets"/"mezcla_v1"
cuenta, n = collections.Counter(), 0
for split in ("train","val"):
    d = raiz/split/"labels"
    if not d.exists(): d = raiz/"labels"/split
    for t in d.glob("*.txt"):
        n += 1
        for l in t.read_text().split("\n"):
            if l.strip(): cuenta[int(l.split()[0])] += 1

print(f"{n} etiquetas\n")
faltan = []
for i, nom in enumerate(CLASES):
    k = cuenta.get(i, 0)
    marca = ""
    if k == 0:
        marca = "   <-- VACÍA" + ("  (IMPRESCINDIBLE)" if nom in IMPRESCINDIBLES else "  (ok sin D-Fire)")
        if nom in IMPRESCINDIBLES: faltan.append(nom)
    print(f"  {i} {nom:<10} {k:>7} cajas{marca}")
if faltan:
    raise SystemExit(f"\nSin cajas de {faltan}. Entrenar así le enseña al modelo "
                     f"que no existen. Revisá bajar_datasets.py antes de seguir.")
print("\n✓ las clases que desbloquean tracking, caídas, agresión y ReID tienen datos")
!python diagnostico_clases.py

## 7 · Entrenar

Solo la cabeza: el backbone va congelado y en `no_grad`.

**No uses `--cachear-features`.** Aprovecha que el backbone está congelado,
pero cuesta ~3 MB por foto: 13.000 imágenes son ~39 GB.

Si la sesión se corta, volvé a correr: retoma desde `/kaggle/working`.

In [ ]:
import os
EPOCAS, BATCH, TAM = 60, 8, 384

# los checkpoints van al output persistente, no al scratch
ck_local = f"{OBJ}/checkpoints"
if os.path.islink(ck_local): os.unlink(ck_local)
elif os.path.exists(ck_local): shutil.rmtree(ck_local)
os.symlink(CKPT, ck_local)

flag = f"--reanudar {CKPT}/head_last.pt" if os.path.exists(f"{CKPT}/head_last.pt") else ""
print("retomando" if flag else "empezando de cero")

%cd {OBJ}
!python entrenar_objetos_cuda.py --dataset datasets/mezcla_v1 \
    --epocas {EPOCAS} --batch {BATCH} --tam {TAM} --precision bf16 {flag}

## 8 · Incrustar el backbone y verificar

Acá se cierra el agujero que costó el v2. Los tensores irreconstruibles —FPN,
`embed_head`, GRU: 3,4 M de parámetros, 13,6 MB— se meten **adentro** del
checkpoint, y queda con huella.

La verificación no es de mentira: se copia el `.pt` solo a otra carpeta, **sin**
el `backbone.pt` al lado, y se carga el motor desde ahí.

In [ ]:
%cd {OBJ}
!python exportar_objetos.py --incrustar-backbone checkpoints/head_best.pt \
    --backbone checkpoints/backbone.pt --salida checkpoints/head_best_solo.pt

import tempfile, shutil, torch, os
solo = f"{CKPT}/head_best_solo.pt"
tmp = tempfile.mkdtemp(); shutil.copy(solo, tmp)          # sin backbone.pt al lado
from objects_engine import EngineConfig, ObjectsEngine
ObjectsEngine(EngineConfig(pesos=os.path.join(tmp, "head_best_solo.pt"), verboso=True))

ck = torch.load(solo, map_location="cpu", weights_only=False)
m = (ck.get("metricas") or {}).get("mAP@0.50")
print(f"\nhuella  : {ck['backbone_huella']}")
print(f"época   : {ck.get('epoca')}")
print(f"mAP@0.50: {m}   (v2 de referencia: 0.5335)")
print(f"tamaño  : {os.path.getsize(solo)/1e6:.1f} MB")

## 9 · Bajarlo

Queda en el **Output** del notebook. Bajalo y ponelo en
`horus/04_cabezas/objetos/modelos/`.

De ahí en más el motor lo carga solo, verifica la huella al arrancar, y si
algún día no coincide **corta** en vez de detectar mal en silencio.

In [ ]:
import os, shutil
for f in ("head_best_solo.pt", "head_best.pt", "backbone.pt", "historial.json"):
    p = os.path.join(CKPT, f)
    if os.path.exists(p):
        print(f"  {f:<22} {os.path.getsize(p)/1e6:>7.1f} MB")
print("\nEstán en el panel Output -> checkpoints/")
print("El que necesitás es head_best_solo.pt. Los otros son respaldo.")